In [40]:
import os, json
import pandas as pd


# analyse the steering results
success schemas:
- counting 1s and -1s
    - counts of 1s of 2pos
    - counts of -1s of 2neg
- comparing the baseline
    - if baseline is 1, after 2neg, how many becomes -1 or 0?
    - if baseline is 0, 
        - after 2neg, how many becomes -1?
        - after 2pos, how many becomes 1?
    - if baseline is -1, after 2pos, how many becomes 0 or 1?
- counts of 1s for bridge

additionally for quality
- counts of 1s for repetition
- average fluency

# baseline
no generated sentence in the baseline is talking about the golden gate bridge

In [ ]:
# baseline files have a different structure
def base_stats(dir, base_file):
    """
    return the sentiment labels of the base generation for downstream process
    """
    print("analysing ", base_file)
    df = pd.read_json(dir + base_file)
    print("counts of", df["continuation_label"].value_counts())
    print("number of repetitive sentences:", df["repetition"].sum().item())
    print("average perplexity of continuations:", df["fluency"].mean().item())
    print()
    return df["continuation_label"]


analysing  gemini_base_llama_senti+_fl_temp_0.json
counts of continuation_label
 0    13
 1     5
-1     2
Name: count, dtype: int64
number of repetitive sentences: 5
average perplexity of continuations: 2.557923251390457

analysing  gemini_base_opt_senti+_fl_temp_0.json
counts of continuation_label
 0    9
 1    7
-1    4
Name: count, dtype: int64
number of repetitive sentences: 12
average perplexity of continuations: 3.263213074207306

analysing  gemini_base_de_senti+_fl.json
counts of continuation_label
0    16
1     4
Name: count, dtype: int64
number of repetitive sentences: 11
average perplexity of continuations: 2.575465887784958

analysing  gemini_base_zh_senti+_fl.json
counts of continuation_label
 0    15
 1     3
-1     2
Name: count, dtype: int64
number of repetitive sentences: 7
average perplexity of continuations: 6.066083538532257



# temperature=1
all the results above are for when temperature = 0
below are additional analysis for when temperature = 1, which includes
- steering with a pair of phrases with one white space prepended to each phrase
- steering with a pair of sentences

In [ ]:
baseline_path = "/scratch/fmeng/ActAdd/results/gemini_base/"
result_path = "/scratch/fmeng/ActAdd/results/"
baseline_llama_temp1 = "gemini_base_llama_fl_senti.json"
baseline_opt_temp1 = "gemini_base_opt_fl_senti.json"

dirs_llama_temp1 = [
    "gemini_2pos_llama_senti_hpt", 
    "gemini_2neg_llama_senti_hpt", 
    "gemini_sent_2pos_llama_senti_hpt", 
    "gemini_sent_2neg_llama_senti_hpt"
    ]

dirs_opt_temp1 = [
    "gemini_2pos_opt_senti_hpt", 
    "gemini_2neg_opt_senti_hpt", 
    "gemini_sent_2pos_opt_senti_hpt",
    "gemini_sent_2neg_opt_senti_hpt"
    ]

In [28]:
def base_temp1(base_file):
    df = pd.read_json(baseline_path + base_file)
    print("counts of", df["continuation_label"].value_counts())
    print("average perplexity of continuations:", df["fluency"].mean().item())
    return df["continuation_label"]
base_llama_sentimap_temp1 = base_temp1(baseline_llama_temp1)
base_opt_sentimap_temp1 = base_temp1(baseline_opt_temp1)
print(base_llama_sentimap_temp1, base_opt_sentimap_temp1)

counts of continuation_label
 1    11
 0     8
-1     1
Name: count, dtype: int64
average perplexity of continuations: 22.038273668289186
counts of continuation_label
 0    12
 1     6
-1     2
Name: count, dtype: int64
average perplexity of continuations: 27.720147919654845
0     1
1    -1
2     0
3     0
4     1
5     1
6     0
7     1
8     1
9     0
10    1
11    1
12    1
13    0
14    0
15    1
16    1
17    1
18    0
19    0
Name: continuation_label, dtype: int64 0     1
1     1
2     1
3     0
4    -1
5     0
6     0
7     0
8     1
9     0
10    0
11    1
12   -1
13    0
14    0
15    1
16    0
17    0
18    0
19    0
Name: continuation_label, dtype: int64


## sentiment counting 1s or -1s


In [29]:
def senti_temp1_count(dir):
    print("showing result for directory", dir)
    lst_files = os.listdir(f"{result_path}{dir}/")
    n_file = len(lst_files)
    grid_one = pd.DataFrame(0, index=range(n_file),columns=range(20))
    grid_zero = pd.DataFrame(0, index=range(n_file),columns=range(20))
    grid_neg = pd.DataFrame(0, index=range(n_file),columns=range(20))
    for file_name in lst_files:
        layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
        with open(f"{result_path}{dir}/{file_name}", "r") as f: 
            r_dict = json.load(f)
        for coeff in r_dict:
            list_dict = pd.DataFrame(r_dict[coeff])
            coeff = int(coeff)
            if 1 in list_dict["continuation_label"].value_counts():
                grid_one.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[1]
            if 0 in list_dict["continuation_label"].value_counts():
                grid_zero.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[0]
            if -1 in list_dict["continuation_label"].value_counts():
                grid_neg.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[-1]
    if "2pos" in dir:
        print("count of positive continuation ↑")
        display(grid_one.style.background_gradient(cmap='Blues', axis=None))
    elif "2neg" in dir:
        print("count of negative continuation ↑")
        display(grid_neg.style.background_gradient(cmap='Blues', axis=None))

### llama

In [30]:
for dir in dirs_llama_temp1:
    senti_temp1_count(dir)

showing result for directory gemini_2pos_llama_senti_hpt
count of positive continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,7,7,3,3,3,4,4,4,4,5,6,8,7,7,8,6,4,4,6,7
1,7,8,8,5,5,5,4,4,4,4,4,5,6,5,5,5,6,6,6,7
2,5,9,6,6,5,6,6,7,7,7,7,6,6,5,4,4,4,6,6,5
3,6,5,6,8,8,9,9,7,6,6,5,5,4,4,5,5,5,5,6,6
4,6,6,7,7,8,10,9,7,6,7,7,7,7,7,7,5,6,5,4,5
5,5,7,7,6,6,7,7,5,8,8,8,8,9,10,9,9,8,8,7,7
6,6,7,10,9,10,7,6,7,7,7,6,6,5,6,6,7,7,8,9,10
7,5,8,9,8,4,7,5,8,8,8,10,8,7,8,8,6,4,4,4,3
8,5,5,7,7,7,8,7,8,8,11,9,8,9,4,6,6,7,7,7,5
9,5,7,7,8,10,9,7,7,8,8,8,7,6,7,7,7,6,8,8,8


showing result for directory gemini_2neg_llama_senti_hpt
count of negative continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4,0,1,0,1,1,1,1,1,1,1,0,0,1,1,0,0,0,0,0
1,3,3,4,5,4,4,4,3,4,3,3,6,2,3,5,3,3,2,1,0
2,4,5,6,5,3,4,5,1,1,4,1,2,1,2,0,3,2,3,3,2
3,0,3,3,3,2,2,3,5,4,2,2,2,2,4,5,6,2,4,5,6
4,2,2,2,4,4,4,4,3,4,3,2,4,2,2,3,1,1,2,1,1
5,2,1,2,3,3,2,3,2,3,3,3,2,3,4,3,3,2,3,2,3
6,2,1,1,3,2,2,2,2,2,2,2,1,4,4,3,2,2,2,3,3
7,1,2,1,2,4,3,3,3,3,4,3,1,1,1,2,3,4,2,3,2
8,1,3,2,2,3,4,4,4,2,1,1,1,1,3,2,1,1,2,2,2
9,2,2,2,2,2,3,2,1,3,1,3,2,3,3,2,2,4,4,5,4


showing result for directory gemini_sent_2pos_llama_senti_hpt
count of positive continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,7,6,6,6,8,8,8,7,6,6,6,7,7,7,7,8,7,7,8
1,7,6,4,7,8,6,5,6,10,6,6,3,3,5,7,7,10,10,10,9
2,7,5,6,6,6,6,5,2,5,2,1,2,2,1,0,2,2,0,0,3
3,6,3,6,5,7,8,8,9,8,7,9,8,11,11,12,11,11,9,11,8
4,9,5,7,10,5,9,10,10,6,8,9,9,9,6,5,5,8,3,5,5
5,4,6,8,9,10,9,11,8,8,5,4,5,2,1,4,5,4,0,4,2
6,4,8,8,7,10,7,10,12,11,10,7,11,13,13,11,11,13,10,6,8
7,4,8,8,7,9,10,9,7,7,11,11,8,8,6,6,12,10,11,8,6
8,5,7,8,8,8,10,11,9,8,8,6,11,10,7,6,10,13,8,10,12
9,4,5,6,8,7,8,9,7,6,4,5,9,10,11,11,10,9,8,12,12


showing result for directory gemini_sent_2neg_llama_senti_hpt
count of negative continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4,1,3,3,2,1,1,1,1,0,0,0,0,0,0,0,0,0,0,1
1,4,2,4,3,3,2,3,2,1,0,1,0,0,0,1,2,2,3,4,2
2,1,1,2,3,4,3,2,3,1,0,2,2,2,2,1,1,2,1,1,0
3,4,2,2,5,6,5,3,2,2,2,2,1,0,1,2,1,1,1,1,1
4,2,5,3,4,3,3,3,5,4,4,3,3,2,3,1,0,3,5,5,5
5,3,5,4,4,4,4,3,3,2,1,1,2,1,1,2,1,2,2,3,3
6,2,5,5,5,4,4,4,3,3,2,2,5,4,3,3,4,4,3,3,5
7,2,6,5,5,3,3,2,3,4,1,2,2,2,2,3,4,6,3,5,3
8,3,4,3,3,6,4,6,6,4,4,3,2,4,5,7,3,4,4,4,5
9,5,5,5,2,2,3,3,3,4,3,6,4,4,3,3,7,4,4,5,6


### opt

In [31]:
for dir in dirs_opt_temp1:
    senti_temp1_count(dir)

showing result for directory gemini_2pos_opt_senti_hpt
count of positive continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,6,1,4,3,4,3,1,2,1,4,6,7,7,8,7,6,6,6,6,5
1,6,7,7,5,4,5,5,4,1,4,4,6,5,7,3,9,1,2,2,3
2,4,4,10,5,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4,3,3,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
4,4,5,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,3,4,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6,4,6,3,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,5,5,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,6,4,4,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,5,4,4,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


showing result for directory gemini_2neg_opt_senti_hpt
count of negative continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,3,4,5,4,3,3,3,4,2,2,2,3,2,2,4,3,2,2,2
1,3,3,3,4,3,3,4,4,3,3,0,1,1,1,1,2,1,3,1,3
2,3,5,2,5,3,3,0,1,0,0,0,0,0,1,0,0,0,0,0,0
3,3,2,4,2,2,2,3,3,5,2,1,0,0,0,1,1,1,0,0,0
4,3,1,4,2,2,3,2,1,1,0,0,1,0,0,0,0,0,0,0,0
5,3,3,2,1,1,2,3,4,2,1,0,1,0,0,1,0,0,0,0,0
6,3,4,3,3,0,1,1,1,1,3,0,1,0,0,0,0,0,0,0,0
7,3,4,3,3,4,1,4,4,2,1,1,2,1,1,0,0,0,0,0,0
8,1,2,2,3,0,4,2,2,3,0,1,1,0,0,1,0,0,0,0,1
9,2,2,2,4,1,1,0,2,1,1,1,1,1,0,0,0,0,1,1,0


showing result for directory gemini_sent_2pos_opt_senti_hpt
count of positive continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,8,6,5,6,7,9,7,8,7,6,6,6,6,7,7,7,5,5,5,4
1,7,7,5,10,6,9,9,9,10,9,9,6,7,5,7,8,6,6,9,7
2,7,6,5,5,4,3,2,5,0,3,4,3,1,1,1,1,0,0,0,0
3,4,4,3,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0
4,6,6,6,5,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
5,4,7,5,6,3,3,2,0,0,1,0,0,0,0,0,0,0,0,0,0
6,3,6,5,5,1,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0
7,4,5,6,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
8,3,5,4,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
9,4,6,3,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


showing result for directory gemini_sent_2neg_opt_senti_hpt
count of negative continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1,4,4,4,5,6,6,4,3,3,2,1,2,2,2,2,2,2,2,2
1,3,2,3,5,3,3,2,2,1,2,3,1,2,2,3,3,3,2,1,3
2,3,3,4,5,7,3,3,5,2,2,3,1,4,4,3,2,0,3,0,2
3,3,5,6,7,6,2,2,2,0,2,1,1,1,1,2,2,2,1,1,2
4,3,2,5,0,3,1,1,0,0,0,0,0,0,0,0,0,1,1,1,2
5,2,2,4,3,1,2,1,1,2,1,1,4,2,1,5,3,4,2,3,3
6,2,2,4,2,0,2,1,3,2,3,0,1,1,0,0,1,1,1,0,1
7,2,4,0,1,1,2,2,3,2,1,1,2,4,4,1,0,1,0,0,1
8,2,4,2,2,2,0,2,0,0,1,0,0,0,0,0,1,0,0,0,0
9,2,4,5,1,0,2,2,2,3,0,3,2,2,3,0,2,1,2,2,3


## sentiment comparing with the baseline
### llama

In [ ]:
# base_llama_sentimap[10].item()  # 0, 1, -1
def comparative_stats(dir, sentimap, include_fl=False):
    """
    base_llama_sentimap or base_opt_sentimap 
    gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt
    """
    print("showing result for directory", dir)    
    lst_files = os.listdir(f"{result_path}{dir}/")
    n_files = len(lst_files)
    grid_success = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_rep = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_fl = pd.DataFrame(index=range(n_files),columns=range(20))
    if "_2pos" in dir or "__love" in dir or "_Love" in dir:  # count the tags that are larger than the corresponding one in the base_map
        for file_name in lst_files:
            layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
            with open(f"{result_path}{dir}/{file_name}", "r") as f: 
                r_dict = json.load(f)
            for coeff in r_dict:
                list_dict = pd.DataFrame(r_dict[coeff])
                coeff = int(coeff)
                # pointwise compare with llama_sentimap
                successs = list_dict["continuation_label"] > sentimap
                grid_success.loc[layer, coeff-1] = successs.sum()
                grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
                grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    if "_2neg" in dir or "_Hate" in dir or "__hate" in dir:  # count the tags that are smaller than the corresponding one in the base_map
        for file_name in lst_files:
            layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
            with open(f"{result_path}{dir}/{file_name}", "r") as f: 
                r_dict = json.load(f)
            for coeff in r_dict:
                list_dict = pd.DataFrame(r_dict[coeff])
                coeff = int(coeff)
                successs = list_dict["continuation_label"] < sentimap
                grid_success.loc[layer, coeff-1] = successs.sum()
                grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
                grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    print("count of bridges ↑")
    display(grid_success.style.background_gradient(cmap='Blues', axis=None))
    if not include_fl:
        grid_fl = None
    print("harmonic mean ↑")
    # hms = get_means(grid_success, grid_rep, grid_fl)
    # hms = hms.apply(pd.to_numeric).astype(float)
    # display(hms.style.background_gradient(cmap='Blues', axis=None))


In [32]:
for dir in dirs_llama_temp1:
    comparative_stats(dir, base_llama_sentimap_temp1)

showing result for directory gemini_2pos_llama_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,2,2,2,2,1,1,1,1,1,2,3,3,3,3,2,2,2,2,3
1,2,2,3,1,2,2,2,2,2,2,2,3,3,2,2,2,2,2,2,2
2,3,4,1,2,1,1,2,3,3,3,3,2,2,1,1,1,1,2,2,2
3,2,2,2,2,2,2,2,2,2,2,2,2,1,1,1,1,1,1,2,1
4,2,2,2,2,2,3,2,1,1,2,2,3,3,2,2,2,2,1,1,1
5,2,2,3,1,1,1,1,2,3,3,3,3,3,3,3,3,3,3,2,2
6,2,2,4,4,4,3,3,3,3,4,3,2,2,2,2,2,2,2,3,3
7,2,4,4,3,2,4,3,4,2,3,4,2,2,3,3,2,1,1,1,1
8,2,2,2,3,3,3,3,2,3,4,4,3,4,3,3,3,3,4,4,4
9,2,2,2,3,3,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2


showing result for directory gemini_2neg_llama_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,6,6,9,6,9,10,9,9,9,10,8,7,7,7,6,7,8,9,9,10
1,7,5,6,8,8,9,10,9,11,11,8,10,9,11,8,8,7,7,9,9
2,8,10,9,9,10,11,10,8,6,12,6,9,9,8,8,11,11,12,11,12
3,7,7,9,8,8,6,8,9,10,9,11,8,9,11,11,11,7,7,9,10
4,6,6,6,6,6,6,8,7,9,8,10,12,11,10,10,9,10,9,8,9
5,6,5,5,6,7,7,8,7,7,7,9,8,10,9,11,11,8,8,8,8
6,4,5,6,7,7,8,8,8,8,8,7,6,8,10,9,9,11,10,8,8
7,5,6,6,5,8,7,7,8,8,9,9,6,8,7,5,6,8,8,8,9
8,4,6,7,7,7,8,8,8,5,7,10,11,9,8,9,7,5,5,5,5
9,4,5,7,7,7,8,7,6,6,7,8,8,8,8,7,8,9,9,9,8


showing result for directory gemini_sent_2pos_llama_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,1,2,3,3,3,2,2,2,2,2,2,2,2,3,3,3,3,3,3
1,2,2,1,3,3,3,2,2,3,2,3,1,2,3,3,3,3,3,3,3
2,2,2,2,2,2,3,2,1,3,2,1,2,2,2,1,2,1,1,1,2
3,3,1,3,3,4,4,3,4,3,3,4,2,4,4,5,4,3,3,4,4
4,4,2,2,4,3,6,5,3,3,3,4,4,4,3,2,3,3,1,2,2
5,2,2,3,1,3,3,4,4,5,3,3,4,2,2,2,2,2,1,2,2
6,2,2,3,2,3,2,5,6,4,6,5,5,5,5,3,6,6,5,3,5
7,2,4,2,2,4,4,3,2,3,6,5,4,4,3,3,8,5,5,3,3
8,2,2,3,4,3,3,5,3,4,4,3,5,4,3,3,4,6,4,5,6
9,2,2,3,2,1,2,2,2,2,2,2,6,6,4,4,4,2,3,4,5


showing result for directory gemini_sent_2neg_llama_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,8,4,8,10,9,7,7,8,5,5,4,5,7,7,7,7,7,6,7,6
1,9,6,9,8,8,5,4,8,5,6,8,7,5,6,8,9,8,8,8,7
2,7,6,6,5,7,7,4,6,5,3,6,8,10,10,8,8,10,9,10,10
3,6,6,8,10,9,11,9,5,6,6,6,5,5,5,6,6,6,6,6,7
4,5,7,7,9,7,7,8,10,8,11,8,9,9,9,9,7,8,11,9,7
5,8,9,8,10,9,9,11,8,6,7,8,8,7,8,6,6,6,8,9,8
6,4,10,11,11,9,10,12,12,7,5,6,8,10,7,10,10,12,10,9,9
7,7,11,11,11,10,8,7,9,8,8,10,11,10,12,14,13,16,13,13,14
8,8,8,10,9,9,11,12,10,9,9,8,10,10,12,14,10,11,13,11,10
9,10,8,9,9,7,6,6,9,8,9,11,12,12,9,9,15,12,13,13,13


### opt

In [33]:
for dir in dirs_opt_temp1:
    comparative_stats(dir, base_opt_sentimap_temp1)

showing result for directory gemini_2pos_opt_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,3,3,3,3,3,3,4,3,4,5,6,6,7,6,5,4,4,4,3
1,5,6,6,4,4,5,4,5,2,2,3,4,6,7,5,7,3,3,2,3
2,4,5,10,5,3,2,2,2,2,2,1,2,2,2,2,2,2,2,1,2
3,4,2,4,2,2,2,2,2,2,2,1,0,1,2,1,2,2,2,2,2
4,4,6,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
5,4,5,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
6,4,6,3,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
7,5,4,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
8,6,4,4,2,1,2,2,2,1,2,2,2,2,2,2,2,2,2,1,1
9,4,4,5,2,2,2,0,1,1,1,2,2,1,0,1,1,1,1,1,1


showing result for directory gemini_2neg_opt_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,6,8,6,8,7,7,6,8,6,7,7,8,5,5,7,6,5,6,5
1,4,6,6,8,7,6,7,9,7,4,4,4,6,6,6,6,4,5,6,6
2,4,8,5,9,6,8,6,7,6,6,6,6,6,6,6,6,6,6,6,5
3,5,3,6,5,7,5,7,9,11,7,7,6,6,6,6,6,6,6,6,6
4,5,4,6,6,6,7,8,6,6,5,5,6,6,6,6,6,5,6,6,6
5,5,4,4,4,3,5,7,9,8,6,6,7,6,5,6,4,4,5,4,3
6,5,7,6,5,5,6,6,7,7,8,5,6,6,6,6,6,6,6,6,6
7,4,5,6,6,8,7,8,8,7,7,6,6,7,7,6,6,6,6,6,6
8,3,6,4,7,5,9,6,7,7,6,7,6,6,6,6,6,6,6,6,7
9,3,5,3,7,4,7,6,8,7,5,5,6,6,6,6,6,6,7,7,6


showing result for directory gemini_sent_2pos_opt_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,6,6,2,3,5,6,6,6,6,5,5,5,5,5,6,6,5,5,5,4
1,6,6,5,7,5,7,7,7,10,8,7,5,7,3,5,7,6,7,8,6
2,6,5,4,5,4,3,3,4,1,4,4,2,2,2,2,3,2,2,2,2
3,4,4,4,3,2,2,1,1,2,1,1,2,2,2,2,2,2,2,2,2
4,6,6,5,4,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
5,4,6,4,5,3,3,4,2,1,3,2,2,2,2,2,2,2,2,2,2
6,3,6,4,5,2,3,2,2,2,2,2,1,1,2,2,2,2,3,3,2
7,4,5,5,2,2,1,1,1,0,1,1,1,2,1,2,1,1,1,1,1
8,4,6,4,1,3,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
9,5,6,3,3,2,2,2,2,2,2,2,2,2,2,2,1,2,2,2,2


showing result for directory gemini_sent_2neg_opt_senti_hpt


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,3,4,6,6,7,6,7,5,4,5,7,5,5,6,6,6,4,4,4,5
1,5,4,5,7,6,7,5,5,3,4,5,7,7,7,6,6,5,5,6,7
2,4,5,7,9,9,6,7,7,6,8,9,5,7,6,8,6,3,9,6,8
3,6,7,8,8,6,6,6,7,6,7,6,7,6,6,6,7,6,6,7,7
4,6,6,6,4,5,6,4,4,6,5,5,5,5,6,6,6,7,6,7,7
5,5,6,7,7,5,6,6,7,6,7,6,8,7,7,9,8,8,7,8,9
6,5,5,6,6,5,7,7,9,7,7,5,6,6,6,6,6,7,6,6,6
7,5,6,5,5,5,7,6,6,5,6,7,6,8,7,6,6,6,6,6,6
8,5,5,6,6,6,6,8,6,6,6,5,6,6,6,6,6,6,6,6,6
9,6,7,8,7,6,5,7,7,8,6,9,7,8,8,6,8,7,7,7,7
